# Question 1: Install Spark and PySpark

In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/03/09 16:29:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.version

'3.3.2'

# Question 2: Yellow October 2024

In [4]:
# download the file
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-03-09 13:50:36--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.230.209.72, 54.230.209.200, 54.230.209.126, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.209.72|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202 100%[===================>]  61.36M  82.1MB/s    in 0.7s    

2025-03-09 13:50:37 (82.1 MB/s) - ‘yellow_tripdata_2024-10.parquet’ saved [64346071/64346071]



In [20]:
#read into a spark dataframe
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2024-10.parquet')


In [6]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-10-01 00:30:44|  2024-10-01 00:48:26|              1|          3.0|         1|                 N|         162|         246|           1|       18.4|  1.0|    0.5|       1.

In [21]:
#repartition 
df = df.repartition(4)

In [8]:
#save as parquet file
df.write.parquet('homework/output/')

In [13]:
! ls homework/output

_SUCCESS
part-00000-4b0603b9-b900-428d-aa4e-436d204273fa-c000.snappy.parquet
part-00001-4b0603b9-b900-428d-aa4e-436d204273fa-c000.snappy.parquet
part-00002-4b0603b9-b900-428d-aa4e-436d204273fa-c000.snappy.parquet
part-00003-4b0603b9-b900-428d-aa4e-436d204273fa-c000.snappy.parquet


In [14]:
# check average size of the parquet files
!du -sh homework/output/*

0	homework/output/_SUCCESS
25M	homework/output/part-00000-4b0603b9-b900-428d-aa4e-436d204273fa-c000.snappy.parquet
25M	homework/output/part-00001-4b0603b9-b900-428d-aa4e-436d204273fa-c000.snappy.parquet
25M	homework/output/part-00002-4b0603b9-b900-428d-aa4e-436d204273fa-c000.snappy.parquet
25M	homework/output/part-00003-4b0603b9-b900-428d-aa4e-436d204273fa-c000.snappy.parquet


# Question 3: Count records


In [23]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampType(), True), StructField('tpep_dropoff_datetime', TimestampType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True)])

### PySpark

In [25]:
from pyspark.sql import functions as F

In [26]:
q3_result = df.filter(F.to_date(df['tpep_pickup_datetime']) == '2024-10-15').count()

In [27]:
q3_result

128893

### Spark SQL

In [28]:
df.registerTempTable('oct_data')

/home/anqi0607/spark/spark-3.3.2-bin-hadoop3/python/pyspark/sql/dataframe.py:229: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [29]:
spark.sql("""
select count(1)
from oct_data
where cast(tpep_pickup_datetime as date) == '2024-10-15'
""").show()

+--------+
|count(1)|
+--------+
|  128893|
+--------+



# Question 4: Longest trip

### PySpark

In [31]:
duration = df.withColumn('duration',
                        F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime'))

In [33]:
max_duration = duration.agg(F.max('duration')).collect()[0][0]

In [34]:
q4_result = max_duration / 3600

In [35]:
q4_result

162.61777777777777

### Spark SQL

In [38]:
spark.sql("""
select 
    max((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600) as q4_result
from 
    oct_data
""").show()

+------------------+
|         q4_result|
+------------------+
|162.61777777777777|
+------------------+



# Question 5: User Interface

Spark’s User Interface which shows the application's dashboard runs on which local port?

Answer: port 4040


# Question 6: Least frequent pickup location zone

In [39]:
# zone look up data
zones = spark.read \
    .option('header', 'true') \
    .csv('taxi_zone_lookup.csv')


In [40]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



### PySpark

In [44]:
df_join = df.join(zones, df['PULocationID'] == zones['LocationID'], how='inner') \
    .select(df['PULocationID'], zones['Zone'])

In [51]:
q6_result = df_join.groupBy('Zone') \
    .count() \
    .orderBy('count') \
    .first()

In [52]:
q6_result

Row(Zone="Governor's Island/Ellis Island/Liberty Island", count=1)

### Spark SQL

In [54]:
zones.registerTempTable('zones')

In [56]:
spark.sql("""
select
    z.Zone,
    count(1) as frequency
from
    oct_data o join zones z on o.PULocationID = z.LocationID
group by 1
order by 2
limit 1
""").show()

+--------------------+---------+
|                Zone|frequency|
+--------------------+---------+
|Governor's Island...|        1|
+--------------------+---------+

